# **ChromaDB Vector Database Tutorial**

This notebook teaches ChromaDB from first principles, then connects it to LangChain and SQL-agent few-shot example selection.

You will learn:
- collections, IDs, documents, metadata, embeddings
- adding, querying, filtering, updating, deleting
- persistence
- vector similarity and distance
- LangChain `Chroma`
- semantic selection of SQL examples


## 1. The mental model

A relational database usually answers questions such as:

```text
Find rows where brand = 'Levi'
```

A vector database answers questions closer to:

```text
Find records whose meaning is closest to this text.
```

Conceptually:

```text
text -> embedding model -> vector -> ChromaDB -> nearest vectors
```

A typical Chroma record contains:
- an ID
- document text
- metadata
- an embedding vector

In [1]:
import chromadb
import sklearn
import langchain_core

print('chromadb:', chromadb.__version__)
print('scikit-learn:', sklearn.__version__)
print('langchain-core:', langchain_core.__version__)

chromadb: 1.5.9
scikit-learn: 1.7.2
langchain-core: 1.5.4


## 2. Create an in-memory Chroma client

`chromadb.Client()` is convenient for experimentation. Its data is temporary.

In [2]:
client = chromadb.Client()
collection = client.get_or_create_collection(name='tshirt_knowledge')

print('Collection:', collection.name)
print('Count:', collection.count())

Collection: tshirt_knowledge
Count: 0


## 3. Prepare documents and metadata

We use T-shirt examples so the lesson connects to your SQL-agent project.

In [3]:
documents = [
    'Levi sells casual white cotton t-shirts.',
    'Nike has black sports t-shirts designed for training.',
    'Adidas offers blue performance shirts for athletes.',
    'Levi white shirts are available in multiple sizes.',
    'Nike running apparel includes lightweight tops.',
    'Discounts can reduce the final selling price of shirts.'
]

ids = [f'doc{i}' for i in range(1, len(documents) + 1)]

metadatas = [
    {'brand': 'Levi', 'color': 'White', 'type': 'product'},
    {'brand': 'Nike', 'color': 'Black', 'type': 'product'},
    {'brand': 'Adidas', 'color': 'Blue', 'type': 'product'},
    {'brand': 'Levi', 'color': 'White', 'type': 'inventory'},
    {'brand': 'Nike', 'color': 'Unknown', 'type': 'sports'},
    {'brand': 'Generic', 'color': 'Unknown', 'type': 'discount'},
]

for i, doc in enumerate(documents):
    print(ids[i], '->', doc, metadatas[i])

doc1 -> Levi sells casual white cotton t-shirts. {'brand': 'Levi', 'color': 'White', 'type': 'product'}
doc2 -> Nike has black sports t-shirts designed for training. {'brand': 'Nike', 'color': 'Black', 'type': 'product'}
doc3 -> Adidas offers blue performance shirts for athletes. {'brand': 'Adidas', 'color': 'Blue', 'type': 'product'}
doc4 -> Levi white shirts are available in multiple sizes. {'brand': 'Levi', 'color': 'White', 'type': 'inventory'}
doc5 -> Nike running apparel includes lightweight tops. {'brand': 'Nike', 'color': 'Unknown', 'type': 'sports'}
doc6 -> Discounts can reduce the final selling price of shirts. {'brand': 'Generic', 'color': 'Unknown', 'type': 'discount'}


## 4. Create embeddings locally with TF-IDF

A production system normally uses a neural embedding model. We start with TF-IDF because it requires no API key and no model download.

This teaches an important fact: **Chroma stores vectors, but it does not care how those vectors were created.**

TF-IDF is lexical rather than strongly semantic, so later we discuss transformer embeddings.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

vectorizer = TfidfVectorizer()
doc_matrix = normalize(vectorizer.fit_transform(documents))
embeddings = doc_matrix.toarray().tolist()

print('Documents:', len(embeddings))
print('Embedding dimensions:', len(embeddings[0]))
print('First vector, first 10 values:', embeddings[0][:10])

Documents: 6
Embedding dimensions: 36
First vector, first 10 values: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.4658815885508596, 0.4658815885508596]


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = model.encode(documents, normalize_embeddings=True).tolist()

print("Documents:", len(embeddings))
print("Embedding dimensions:", len(embeddings[0]))
print("First vector, first 10 values:", embeddings[0][:10])

W0917 14:06:30.297000 94148 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0917 14:06:30.505000 94148 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.



Documents: 6
Embedding dimensions: 384
First vector, first 10 values: [-0.03386900946497917, 0.04808277636766434, 0.01213289238512516, 0.008410713635385036, 0.021598363295197487, 0.004740310367196798, 0.0977357029914856, 0.02510111965239048, -0.028332818299531937, -0.009446498937904835]


## 5. Add records to Chroma

`add()` inserts records. IDs must be unique inside the collection.

In [6]:
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings,
)

print('Count:', collection.count())

Count: 6


## 6. Inspect records

In [8]:
collection.peek(limit=3)

{'ids': ['doc1', 'doc2', 'doc3'],
 'embeddings': array([[-0.03386901,  0.04808278,  0.01213289, ..., -0.11085638,
         -0.04188597, -0.04285522],
        [-0.02899041,  0.03602462, -0.05787527, ..., -0.12755136,
         -0.04319189, -0.01468045],
        [-0.01184118,  0.02658015, -0.01179878, ..., -0.0924222 ,
          0.03372052,  0.01178084]], shape=(3, 384)),
 'documents': ['Levi sells casual white cotton t-shirts.',
  'Nike has black sports t-shirts designed for training.',
  'Adidas offers blue performance shirts for athletes.'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'brand': 'Levi', 'type': 'product', 'color': 'White'},
  {'color': 'Black', 'brand': 'Nike', 'type': 'product'},
  {'type': 'product', 'brand': 'Adidas', 'color': 'Blue'}]}

In [9]:
collection.get(ids=['doc1', 'doc4'])

{'ids': ['doc1', 'doc4'],
 'embeddings': None,
 'documents': ['Levi sells casual white cotton t-shirts.',
  'Levi white shirts are available in multiple sizes.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'color': 'White', 'type': 'product', 'brand': 'Levi'},
  {'brand': 'Levi', 'color': 'White', 'type': 'inventory'}]}

## 7. Similarity search

A query must be embedded using the **same embedding system** used for the stored documents.

### Distance vs similarity

Chroma commonly returns a distance. In general:

```text
smaller distance = closer vectors = more similar
```

Do not automatically convert distance into a percentage. That depends on the metric and normalization.

In [10]:
query_text = 'white Levi shirts'
#query_vector = normalize(vectorizer.transform([query_text])).toarray().tolist()        # TF-IDF vectorization
query_vector = model.encode([query_text], normalize_embeddings=True).tolist()           # SentenceTransformer vectorization

results = collection.query(
    query_embeddings=query_vector,
    n_results=3,
    include=['documents', 'metadatas', 'distances'],
)

for rank, (doc, meta, distance) in enumerate(
    zip(results['documents'][0], results['metadatas'][0], results['distances'][0]),
    start=1):

    print(f'Rank {rank}')
    print('Document:', doc)
    print('Metadata:', meta)
    print('Distance:', distance)
    print('-' * 60)

Rank 1
Document: Levi white shirts are available in multiple sizes.
Metadata: {'brand': 'Levi', 'color': 'White', 'type': 'inventory'}
Distance: 0.2944348454475403
------------------------------------------------------------
Rank 2
Document: Levi sells casual white cotton t-shirts.
Metadata: {'type': 'product', 'color': 'White', 'brand': 'Levi'}
Distance: 0.37852901220321655
------------------------------------------------------------
Rank 3
Document: Nike running apparel includes lightweight tops.
Metadata: {'brand': 'Nike', 'type': 'sports', 'color': 'Unknown'}
Distance: 1.0493667125701904
------------------------------------------------------------


## 8. Metadata filtering

Vector search can be combined with structured filters.

In [11]:
results_levi = collection.query(
    query_embeddings=query_vector,
    n_results=3,
    where={'brand': 'Levi'},
    include=['documents', 'metadatas', 'distances'],
)

results_levi

{'ids': [['doc4', 'doc1']],
 'embeddings': None,
 'documents': [['Levi white shirts are available in multiple sizes.',
   'Levi sells casual white cotton t-shirts.']],
 'uris': None,
 'included': ['documents', 'metadatas', 'distances'],
 'data': None,
 'metadatas': [[{'type': 'inventory', 'brand': 'Levi', 'color': 'White'},
   {'type': 'product', 'brand': 'Levi', 'color': 'White'}]],
 'distances': [[0.2944348454475403, 0.37852901220321655]]}

In [12]:
collection.get(where={'type': 'product'})

{'ids': ['doc1', 'doc2', 'doc3'],
 'embeddings': None,
 'documents': ['Levi sells casual white cotton t-shirts.',
  'Nike has black sports t-shirts designed for training.',
  'Adidas offers blue performance shirts for athletes.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'brand': 'Levi', 'type': 'product', 'color': 'White'},
  {'brand': 'Nike', 'color': 'Black', 'type': 'product'},
  {'type': 'product', 'color': 'Blue', 'brand': 'Adidas'}]}

## 9. Update, upsert, and delete

- `update()` changes an existing record.
- `upsert()` inserts if missing or updates if present.
- `delete()` removes records.

In [14]:
collection.get(ids=['doc1'])

{'ids': ['doc1'],
 'embeddings': None,
 'documents': ['Levi sells casual white cotton t-shirts.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'brand': 'Levi', 'color': 'White', 'type': 'product'}]}

In [15]:
updated_doc = 'Levi sells premium white cotton t-shirts.'
#updated_embedding = normalize(vectorizer.transform([updated_doc])).toarray()[0].tolist()
updated_embedding = model.encode([updated_doc], normalize_embeddings=True).tolist()[0]

collection.update(
    ids=['doc1'],
    documents=[updated_doc],
    metadatas=[{'brand': 'Levi', 'color': 'White', 'type': 'premium_product'}],
    embeddings=[updated_embedding],
)

collection.get(ids=['doc1'])

{'ids': ['doc1'],
 'embeddings': None,
 'documents': ['Levi sells premium white cotton t-shirts.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'type': 'premium_product', 'color': 'White', 'brand': 'Levi'}]}

In [16]:
new_doc = 'Puma offers red casual t-shirts.'
#new_embedding = normalize(vectorizer.transform([new_doc])).toarray()[0].tolist()
new_embedding = model.encode([new_doc], normalize_embeddings=True).tolist()[0]

collection.upsert(
    ids=['doc7'],
    documents=[new_doc],
    metadatas=[{'brand': 'Puma', 'color': 'Red', 'type': 'product'}],
    embeddings=[new_embedding],
)

print('After upsert:', collection.count())
collection.delete(ids=['doc7'])
print('After delete:', collection.count())

After upsert: 7
After delete: 6


# Part 2: Persistence

An in-memory client disappears with the process. A persistent client stores data on disk.

In [17]:
from pathlib import Path

persist_path = Path('./chroma_tutorial_db')
persistent_client = chromadb.PersistentClient(path=str(persist_path))
persistent_collection = persistent_client.get_or_create_collection(name='persistent_demo')

persistent_collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings,
)

print('Persistent path:', persist_path.resolve())
print('Count:', persistent_collection.count())

Persistent path: C:\Users\meisa\Desktop\Code\ML\Agent_AI\VectorDB\chroma_tutorial_db
Count: 6


After restarting Python, reconnect with:

In [ ]:
#client = chromadb.PersistentClient(path='./chroma_tutorial_db')
#collection = client.get_collection('persistent_demo')

# Part 3: Chroma through LangChain

LangChain wraps vector stores behind a common interface so they can be used by retrievers, RAG pipelines, agents, and few-shot selectors.

In [25]:
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_chroma import Chroma


# 1. LangChain-compatible SentenceTransformer wrapper
class SentenceTransformerEmbeddings(Embeddings):

    def __init__(self, model_name):
        self.model = SentenceTransformer(
            model_name,
            device="cuda"
        )

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            normalize_embeddings=True
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            text,
            normalize_embeddings=True
        ).tolist()


# 2. Initialize embedding model
lc_embedding = SentenceTransformerEmbeddings(
    "sentence-transformers/all-MiniLM-L6-v2"
)


# 3. Create LangChain documents
lc_docs = [
    Document(page_content=doc, metadata=meta)
    for doc, meta in zip(documents, metadatas)
]


# 4. Initialize Chroma vector store
lc_vectorstore = Chroma(
    collection_name="langchain_tshirts",
    embedding_function=lc_embedding,
)


# 5. Remove existing records
existing = lc_vectorstore.get()

if existing.get("ids"):
    lc_vectorstore.delete(ids=existing["ids"])


# 6. Add documents
lc_vectorstore.add_documents(
    documents=lc_docs,
    ids=ids
)

print("Stored:", len(lc_vectorstore.get()["ids"]))

Stored: 6


In [20]:
lc_vectorstore.get(ids=['doc1'])

{'ids': ['doc1'],
 'embeddings': None,
 'documents': ['Levi sells casual white cotton t-shirts.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'type': 'product', 'brand': 'Levi', 'color': 'White'}]}

## 10. LangChain similarity search and retriever

In [21]:
for doc in lc_vectorstore.similarity_search('I want white Levi clothing', k=3):
    print(doc.page_content)
    print(doc.metadata)
    print('-' * 50)

Levi white shirts are available in multiple sizes.
{'brand': 'Levi', 'color': 'White', 'type': 'inventory'}
--------------------------------------------------
Levi sells casual white cotton t-shirts.
{'color': 'White', 'brand': 'Levi', 'type': 'product'}
--------------------------------------------------
Nike running apparel includes lightweight tops.
{'color': 'Unknown', 'type': 'sports', 'brand': 'Nike'}
--------------------------------------------------


In [22]:
retriever = lc_vectorstore.as_retriever(search_kwargs={'k': 2})

for doc in retriever.invoke('sports shirt for training'):
    print(doc.page_content)

Nike has black sports t-shirts designed for training.
Adidas offers blue performance shirts for athletes.


# Part 4: Chroma in your SQL-agent workflow

This is the most relevant architecture for your current project:

```text
                        new user question
                              ↓
                        embedding
                              ↓
                        Chroma similarity search
                              ↓
                        most relevant few-shot examples
                              ↓
                        FewShotPromptTemplate
                              ↓
                        LLM
                              ↓
                        SQL
```

Instead of stuffing every example into the prompt, Chroma retrieves the examples most similar to the new question.

In [26]:
# 1. Initialize embedding model
sql_embedding = SentenceTransformerEmbeddings("sentence-transformers/all-MiniLM-L6-v2")

# 2. Define SQL examples
sql_examples = [
    {
        "Question": "How many white Levi shirts are in stock?",
        "SQLQuery": "SELECT SUM(stock_quantity) FROM t_shirts WHERE brand='Levi' AND color='White';",
        "SQLResult": "[(202,)]",
        "Answer": "There are 202 white Levi shirts in stock.",
    },
    {
        "Question": "How many Nike product variants exist?",
        "SQLQuery": "SELECT COUNT(*) FROM t_shirts WHERE brand='Nike';",
        "SQLResult": "[(12,)]",
        "Answer": "There are 12 Nike product variants.",
    },
    {
        "Question": "What is the total stock of Adidas shirts?",
        "SQLQuery": "SELECT SUM(stock_quantity) FROM t_shirts WHERE brand='Adidas';",
        "SQLResult": "[(350,)]",
        "Answer": "There are 350 Adidas shirts in stock.",
    },
    {
        "Question": "What is the average price of Levi shirts?",
        "SQLQuery": "SELECT AVG(price) FROM t_shirts WHERE brand='Levi';",
        "SQLResult": "[(25.50,)]",
        "Answer": "The average Levi shirt price is 25.50.",
    },
]

# 3. Create LangChain documents
sql_docs = [
    Document(
        page_content=x["Question"],
        metadata={
            "SQLQuery": x["SQLQuery"],
            "SQLResult": x["SQLResult"],
            "Answer": x["Answer"],
        },
    )
    for x in sql_examples
]

# 4. Initialize Chroma with a new collection
sql_store = Chroma(
    collection_name="sql_fewshot_sentence_transformer",
    embedding_function=sql_embedding,
)

# 5. Add or update documents using stable IDs
sql_store.add_documents(
    documents=sql_docs,
    ids=[f"sql{i}" for i in range(len(sql_docs))]
)

# 6. Verify stored examples
print("Stored SQL examples:", len(sql_store.get()["ids"]))

Stored SQL examples: 4


In [27]:
new_question = 'How many black Nike shirts do I have?'
selected = sql_store.similarity_search(new_question, k=2)

for i, doc in enumerate(selected, start=1):
    print(f'Selected example {i}')
    print('Question:', doc.page_content)
    print('SQLQuery:', doc.metadata['SQLQuery'])
    print('Answer:', doc.metadata['Answer'])
    print('-' * 60)

Selected example 1
Question: How many white Levi shirts are in stock?
SQLQuery: SELECT SUM(stock_quantity) FROM t_shirts WHERE brand='Levi' AND color='White';
Answer: There are 202 white Levi shirts in stock.
------------------------------------------------------------
Selected example 2
Question: How many Nike product variants exist?
SQLQuery: SELECT COUNT(*) FROM t_shirts WHERE brand='Nike';
Answer: There are 12 Nike product variants.
------------------------------------------------------------


# Summary

```text
              Text
                ↓
              Embedding model
                ↓
              Vector
                ↓
              Chroma collection
                ↓
              Nearest-neighbor search
                ↓
              Relevant documents/examples
```

ChromaDB is **not the LLM**. It does not reason like an LLM.

Its job is primarily to store vectors and retrieve nearby vectors efficiently. Retrieval quality depends on the embedding model, the stored text, metadata, chunking, and search configuration.